In [1]:
import findspark
findspark.init()

from pyspark.conf import SparkConf
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

conf = SparkConf().setAppName("1907").setMaster("local[4]")
spark = SparkSession.builder.config(conf = conf).getOrCreate()
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/08/14 01:28:27 WARN Utils: Your hostname, de24, resolves to a loopback address: 127.0.1.1; using 192.168.0.102 instead (on interface enp0s3)
25/08/14 01:28:27 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/08/14 01:28:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/08/14 01:28:29 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [ ]:
'''
Table: Accounts

+-------------+------+
| Column Name | Type |
+-------------+------+
| account_id  | int  |
| income      | int  |
+-------------+------+
account_id is the primary key (column with unique values) for this table.
Each row contains information about the monthly income for one bank account.
 

Write a solution to calculate the number of bank accounts for each salary category. The salary categories are:

"Low Salary": All the salaries strictly less than $20000.
"Average Salary": All the salaries in the inclusive range [$20000, $50000].
"High Salary": All the salaries strictly greater than $50000.
The result table must contain all three categories. If there are no accounts in a category, return 0.

Return the result table in any order.

The result format is in the following example.

 

Example 1:

Input: 
Accounts table:
+------------+--------+
| account_id | income |
+------------+--------+
| 3          | 108939 |
| 2          | 12747  |
| 8          | 87709  |
| 6          | 91796  |
+------------+--------+
Output: 
+----------------+----------------+
| category       | accounts_count |
+----------------+----------------+
| Low Salary     | 1              |
| Average Salary | 0              |
| High Salary    | 3              |
+----------------+----------------+
Explanation: 
Low Salary: Account 2.
Average Salary: No accounts.
High Salary: Accounts 3, 6, and 8.
'''

In [2]:
data = [
(3,108939),
(2,12747 ),
(8,87709 ),
(6,91796 )
]
schema = ['account_id','income']

In [3]:
df = spark.createDataFrame(data = data, schema = schema)
df.show()

+----------+------+
|account_id|income|
+----------+------+
|         3|108939|
|         2| 12747|
|         8| 87709|
|         6| 91796|
+----------+------+



In [9]:
Low_df = df.where(F.col("income") < 20000)\
           .agg(F.count(F.col("account_id")).alias("accounts_count"))\
           .select(F.lit("Low Salary").alias("category"), F.col("accounts_count"))

avg_df = df.where((F.col("income") >= 20000) & (F.col("income") <= 50000))\
           .agg(F.count(F.col("account_id")).alias("accounts_count"))\
           .select(F.lit("Average Salary").alias("category"), F.col("accounts_count"))

high_df = df.where(F.col("income") >= 50000)\
  .agg(F.count(F.col("account_id")).alias("accounts_count"))\
  .select(F.lit("High Salary").alias("category"), F.col("accounts_count"))

((Low_df).union(avg_df)).union(high_df)\
.show()

+--------------+--------------+
|      category|accounts_count|
+--------------+--------------+
|    Low Salary|             1|
|Average Salary|             0|
|   High Salary|             3|
+--------------+--------------+



## SQL Solution

<pre>
SELECT "Low Salary" as category, count(account_id) as accounts_count
FROM Accounts
WHERE income < 20000
UNION
SELECT "Average Salary" as category, count(account_id) as accounts_count
FROM Accounts
WHERE income >= 20000 and  income <= 50000
UNIOIN 
SELECT "High Salary" as category, count(account_id) as accounts_count
FROM Accounts
WHERE income >= 50000
</pre>